# Sub-industry clustering (full pairwise brute force, sequential, in-memory)

Take every S&P 500 constituent in the GICS Sub-Industry selected by `SUB_INDUSTRY` in the first cell
(currently **Technology Hardware, Storage & Peripherals**, inside Information Technology), load its
company profile from `descriptions/<SYMBOL>.json`, then ask the model **once per unordered pair**
whether the two profiles describe the same industry. The number of companies and the resulting
`C(n,2)` call count are printed by cell 1's loader.

A pair counts as the same industry when the `company_check` noul probability is **>= 0.70**.

After the loop the n x n score matrix is built and clustered with single-link (union-find) at the same
threshold, so transitively connected companies end up in one group.

No threads, no cache files, no written output - every verdict is printed as it completes and the
results stay in memory.

**Prereq:** `export TYPESAFE_API_KEY="..."` before starting the kernel.

Run top-to-bottom: `Kernel -> Restart & Run All`. Expect roughly one minute per 250 calls at the
measured ~0.25 s per call.

To change the universe, edit the two filters in the first cell: `SUB_INDUSTRY = None` compares all 74
companies of `SECTOR`, and `SECTOR = None` drops the sector filter too.

In [ ]:
import os

if not os.environ.get('TYPESAFE_API_KEY', '').strip():
    raise RuntimeError('TYPESAFE_API_KEY env var is not set - export it, then restart the kernel.')

import csv
import itertools
import json
import time
from pathlib import Path

import numpy as np

from typesafe_sdk import Noul, RetryPolicy, TypeSafeClient, TypeSafeError

SECTOR = 'Information Technology'  # None = any sector
SUB_INDUSTRY = 'Technology Hardware, Storage & Peripherals'  # None = any sub-industry within SECTOR
MODEL = 'jev-latest'
THRESHOLD = 0.70  # noul probability at or above this means "same industry"
CSV_PATH = Path('sp500.csv')
DESCRIPTIONS_DIR = Path('descriptions')
MAX_DESCRIPTION_CHARS = 4000
PAIR_LIMIT = None  # None = every pair; set an int to smoke test

print(f'sector: {SECTOR}  sub-industry: {SUB_INDUSTRY}  model: {MODEL}  same industry when noul >= {THRESHOLD}')

## 1. Load the constituents in scope and their profiles

`SECTOR` and `SUB_INDUSTRY` in the first cell pick the universe; `None` means no filter on that field.

In [ ]:
def load_constituents():
    """Rows of sp500.csv matching SECTOR and SUB_INDUSTRY (None = unfiltered), sorted by symbol."""
    rows = []
    with CSV_PATH.open(newline='', encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            if SECTOR is not None and (row.get('GICS Sector') or '').strip() != SECTOR:
                continue
            if SUB_INDUSTRY is not None and (row.get('GICS Sub-Industry') or '').strip() != SUB_INDUSTRY:
                continue
            rows.append({
                'symbol': (row.get('Symbol') or '').strip(),
                'company_name': (row.get('Security') or '').strip(),
                'sector': (row.get('GICS Sector') or '').strip(),
                'sub_industry': (row.get('GICS Sub-Industry') or '').strip(),
            })
    rows.sort(key=lambda row: row['symbol'])
    return rows


def load_descriptions(symbols):
    """Company profile text per symbol; symbols without a usable profile are dropped with a warning."""
    descriptions = {}
    for symbol in symbols:
        path = DESCRIPTIONS_DIR / f'{symbol}.json'
        if not path.exists():
            print(f'[warn] no profile file for {symbol} ({path}) - skipped')
            continue
        profile = json.loads(path.read_text(encoding='utf-8'))
        text = (profile.get('description') or '').strip()
        if not text:
            print(f'[warn] empty description for {symbol} - skipped')
            continue
        if len(text) > MAX_DESCRIPTION_CHARS:
            print(f'[warn] {symbol} description is {len(text)} chars - truncated to {MAX_DESCRIPTION_CHARS}')
            text = text[:MAX_DESCRIPTION_CHARS]
        descriptions[symbol] = text
    return descriptions


constituents = load_constituents()
meta = {row['symbol']: row for row in constituents}
descriptions = load_descriptions([row['symbol'] for row in constituents])
symbols = sorted(descriptions)
all_pairs = list(itertools.combinations(symbols, 2))

assert len(all_pairs) == len(symbols) * (len(symbols) - 1) // 2, 'pair count must be C(n,2)'
print(f'constituents: {len(constituents)}  with profiles: {len(symbols)}  pairs: {len(all_pairs)} (= C({len(symbols)},2))')
scope = sorted({row['sector'] + ' / ' + row['sub_industry'] for row in meta.values()})
print(f'scope: {"; ".join(scope)}')
lengths = [len(descriptions[symbol]) for symbol in symbols]
print(f'description length: min {min(lengths)}  median {int(np.median(lengths))}  max {max(lengths)} chars')

## 2. Prompt payload

`state` carries the two company profiles (`baseline_company` first, `check_company` second) and the
`company_check` noul question is a yes/no judgement of "same industry".

In [ ]:
def build_state(baseline_company, check_company):
    return {"baseline_company": baseline_company, "check_company": check_company}


def build_questions():
    return {
        "company_check": Noul(
            instructions=(
                "check the 'check_company' in state to determine if it is in the same "
                "industry as 'baseline_company'"
            ),
            criteria={
                "true": "check_company is the same industry as baseline_company",
                "false": "check_company is in different industry compare to baseline_company",
            },
        )
    }

## 3. One comparison

One `system_one` call per pair, with the SDK's own retry/backoff on the client. A failed pair raises
here and is recorded as skipped by the loop below - it is never treated as "same industry".

`MODEL = 'jev-latest'` is an alias: the API resolves it to a concrete version and reports that in
`response.model`.

In [ ]:
client = TypeSafeClient(
    model=MODEL,
    retry=RetryPolicy(max_retries=4, backoff_initial=1.0, backoff_max=10.0, timeout=120.0),
    timeout=120.0,
)

try:
    print(f'models available: {[m.name for m in client.models.list().models]}')
except TypeSafeError as exc:
    print(f'(model list unavailable: {exc})')


def judge_same_industry(baseline_description, check_description):
    """Return the noul score, the >= threshold verdict, the model used, and token usage."""
    try:
        response = client.system_one(
            state=build_state(baseline_description, check_description),
            questions=build_questions(),
        )
    except TypeSafeError as exc:
        raise RuntimeError(f'TypeSafe call failed: {exc}') from exc
    answer = response.nouls.get('company_check')
    if answer is None:
        raise RuntimeError("TypeSafe response contained no 'company_check' noul answer")
    noul = float(answer.noul)
    return {
        'noul': noul,
        'same_industry': noul >= THRESHOLD,
        'model': response.model,
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
    }


demo_a, demo_b = symbols[0], symbols[1]
demo = judge_same_industry(descriptions[demo_a], descriptions[demo_b])
print(
    f"{demo_a} vs {demo_b} -> same_industry={demo['same_industry']}  noul={demo['noul']:.4f}  "
    f"model={MODEL} -> {demo['model']}  "
    f"tokens in={demo['input_tokens']} out={demo['output_tokens']}"
)

## 4. Full pairwise brute force

Alphabetical pair order: the earlier symbol is always `baseline_company`, the later one
`check_company`. One line per verdict as it lands.

In [ ]:
def run_pairwise(pairs, pair_limit=None):
    """Compare every pair sequentially, printing each verdict. Errors are recorded, never fatal."""
    total = len(pairs) if pair_limit is None else min(pair_limit, len(pairs))
    results = []
    started = time.time()
    for index, (baseline_symbol, check_symbol) in enumerate(pairs[:total], start=1):
        try:
            verdict = judge_same_industry(descriptions[baseline_symbol], descriptions[check_symbol])
        except (RuntimeError, TypeSafeError) as exc:
            print(f'{index:04d}/{total} {baseline_symbol}-{check_symbol} [skip] {exc}')
            results.append({
                'a': baseline_symbol, 'b': check_symbol, 'noul': None,
                'same_industry': None, 'model': None, 'error': str(exc),
            })
            continue
        results.append({
            'a': baseline_symbol, 'b': check_symbol, 'noul': verdict['noul'],
            'same_industry': verdict['same_industry'], 'model': verdict['model'], 'error': None,
        })
        label = 'SAME' if verdict['same_industry'] else 'DIFF'
        print(f"{index:04d}/{total} {baseline_symbol}-{check_symbol} noul={verdict['noul']:.4f} {label}")
        if index % 25 == 0 or index == total:
            print(f'  ...{index}/{total} pairs in {(time.time() - started) / 60.0:.1f} min')
    return results


results = run_pairwise(all_pairs, PAIR_LIMIT)

same_count = sum(1 for r in results if r['same_industry'] is True)
different_count = sum(1 for r in results if r['same_industry'] is False)
skipped_count = sum(1 for r in results if r['same_industry'] is None)
print(f'pairs: {len(results)}  same: {same_count}  different: {different_count}  skipped: {skipped_count}')
assert same_count + different_count + skipped_count == len(results)

## 5. Score matrix + single-link groups

Edges are the pairs judged >= `THRESHOLD`; union-find over those edges turns the verdicts into
transitively connected industry groups.

In [ ]:
def build_matrix(symbols, results):
    """Symmetric n x n score matrix; NaN on the diagonal and for pairs that were skipped."""
    index = {symbol: i for i, symbol in enumerate(symbols)}
    matrix = np.full((len(symbols), len(symbols)), np.nan)
    for row in results:
        if row['noul'] is None:
            continue
        i, j = index[row['a']], index[row['b']]
        matrix[i, j] = matrix[j, i] = row['noul']
    return matrix


def single_link_groups(matrix, symbols, threshold):
    """Union-find over every pair scoring >= threshold; groups sorted by size, then name."""
    parent = list(range(len(symbols)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        root_x, root_y = find(x), find(y)
        if root_x != root_y:
            parent[max(root_x, root_y)] = min(root_x, root_y)

    rows, cols = np.where(~np.isnan(matrix) & (matrix >= threshold))
    for i, j in zip(rows, cols):
        if i < j:
            union(int(i), int(j))

    buckets = {}
    for i, symbol in enumerate(symbols):
        buckets.setdefault(find(i), []).append(symbol)
    groups = [sorted(members) for members in buckets.values()]
    groups.sort(key=lambda members: (-len(members), members[0]))
    return groups


def group_stats(group, matrix, symbols):
    """min / mean / max noul across the compared pairs inside one group."""
    index = {symbol: i for i, symbol in enumerate(symbols)}
    scores = [matrix[index[a], index[b]] for i, a in enumerate(group) for b in group[i + 1:]]
    scores = [s for s in scores if not np.isnan(s)]
    if not scores:
        return None
    return min(scores), float(np.mean(scores)), max(scores)


matrix = build_matrix(symbols, results)
groups = single_link_groups(matrix, symbols, THRESHOLD)

assert matrix.shape == (len(symbols), len(symbols)), 'matrix must be square over every symbol'
off_diagonal = matrix[~np.eye(len(symbols), dtype=bool)]
assert np.allclose(off_diagonal, off_diagonal.T, equal_nan=True), 'matrix must be symmetric'
assert sorted(s for g in groups for s in g) == sorted(symbols), 'groups must partition every symbol'

for number, group in enumerate(groups, start=1):
    if len(group) == 1:
        continue
    stats = group_stats(group, matrix, symbols)
    score_line = 'no compared pairs' if stats is None else f'noul min {stats[0]:.4f}  mean {stats[1]:.4f}  max {stats[2]:.4f}'
    print(f'GROUP {number} (n={len(group)})  {score_line}')
    for symbol in group:
        print(f'    {symbol:5s} {meta[symbol]["company_name"]} ({meta[symbol]["sub_industry"]})')
    print(f'    sub-industries: {"; ".join(sorted({meta[s]["sub_industry"] for s in group}))}')

singletons = [group[0] for group in groups if len(group) == 1]
print(f'\nsingleton groups ({len(singletons)}): {" ".join(singletons)}')
print(f'total groups: {len(groups)}  symbols: {len(symbols)}  same-industry pairs: {sum(1 for r in results if r["same_industry"] is True)}  skipped pairs: {sum(1 for r in results if r["same_industry"] is None)}')